# Module 11: Memory & Context Management

**Goal:** Build a memory-enabled assistant that remembers users across sessions — short-term, long-term, and hierarchical.

**What you'll learn:**
1. Why LLMs are stateless and what that means
2. Short-term memory: conversation buffers and token budgeting
3. Long-term memory: vector store of past interactions
4. Working memory: assembling the right context for each request
5. Hierarchical memory (L1/L2/L3)
6. Memory compression to fight context bloat

---

## 0. Setup

In [ ]:
%pip install -q openai chromadb sentence-transformers tiktoken python-dotenv

In [ ]:
import os, json, time, shutil
from datetime import datetime
from collections import deque
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import tiktoken
from openai import OpenAI

client = OpenAI()
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def chat(messages: list[dict], model="gpt-4o-mini") -> str:
    return client.chat.completions.create(model=model, messages=messages, temperature=0.7).choices[0].message.content

print("✅ Ready")

---
## 1. The Statelessness Problem

Every LLM call is stateless — the model has zero memory of previous calls:

```python
chat([{"role": "user", "content": "My name is Alice"}])  
# → "Nice to meet you, Alice!"

chat([{"role": "user", "content": "What's my name?"}])   
# → "I don't know your name."  ← no memory!
```

**Memory** = the mechanism we build to re-inject relevant past context into each new call.

In [ ]:
# Demonstrate the problem
r1 = chat([{"role": "user", "content": "My name is Alice and I love Python."}])
print("Turn 1:", r1)

r2 = chat([{"role": "user", "content": "What's my name and favourite language?"}])
print("Turn 2 (no memory):", r2)

---
## 2. Short-Term Memory: Conversation Buffer

In [ ]:
class ConversationBuffer:
    """
    Sliding window of conversation history.
    Trims oldest turns when token budget is exceeded.
    """

    def __init__(self, max_tokens: int = 2000, system_prompt: str = "You are a helpful assistant."):
        self.max_tokens = max_tokens
        self.system = {"role": "system", "content": system_prompt}
        self._turns: deque[dict] = deque()

    def add(self, role: str, content: str):
        self._turns.append({"role": role, "content": content})
        self._trim()

    def _trim(self):
        """Drop oldest user/assistant pairs until within budget."""
        while self._token_count() > self.max_tokens and len(self._turns) >= 2:
            self._turns.popleft()   # Drop oldest user
            if self._turns:
                self._turns.popleft()  # Drop matching assistant

    def _token_count(self) -> int:
        all_text = " ".join(m["content"] for m in [self.system] + list(self._turns))
        return count_tokens(all_text)

    def messages(self) -> list[dict]:
        return [self.system] + list(self._turns)

    def chat(self, user_message: str) -> str:
        self.add("user", user_message)
        response = chat(self.messages())
        self.add("assistant", response)
        return response

    def __repr__(self):
        return f"ConversationBuffer({len(self._turns)} turns, ~{self._token_count()} tokens)"


buf = ConversationBuffer(max_tokens=1000)

turns = [
    "My name is Alice and I'm a Python developer.",
    "I'm working on a RAG application for legal documents.",
    "What's my name?",
    "What project am I working on?",
]

for msg in turns:
    print(f"\nUser: {msg}")
    response = buf.chat(msg)
    print(f"Assistant: {response[:150]}")

print(f"\n{buf}")

---
## 3. Long-Term Memory: Vector Store

In [ ]:
import chromadb
import hashlib

if os.path.exists("./ltm_db"):
    shutil.rmtree("./ltm_db")

chroma = chromadb.PersistentClient(path="./ltm_db")
collection = chroma.get_or_create_collection("long_term_memory")

def embed(text: str) -> list[float]:
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding


class LongTermMemory:
    """Persist and retrieve memories by semantic similarity."""

    def store(self, text: str, metadata: dict = None):
        mem_id = hashlib.md5(text.encode()).hexdigest()[:12]
        collection.upsert(
            ids=[mem_id],
            embeddings=[embed(text)],
            documents=[text],
            metadatas=[{"timestamp": datetime.now().isoformat(), **(metadata or {})}],
        )

    def recall(self, query: str, k: int = 3) -> list[str]:
        results = collection.query(query_embeddings=[embed(query)], n_results=min(k, collection.count()))
        return results["documents"][0] if results["documents"] else []

    def __len__(self):
        return collection.count()


ltm = LongTermMemory()

# Store some memories
memories_to_store = [
    "User is a Python developer named Alice working on RAG applications.",
    "User prefers concise explanations with code examples.",
    "User is building a legal document search tool using ChromaDB.",
    "User mentioned their team uses FastAPI for all backend services.",
    "User had trouble with LangChain version compatibility in January 2025.",
]
for m in memories_to_store:
    ltm.store(m, metadata={"user": "alice"})

print(f"Stored {len(ltm)} memories")

# Recall relevant memories
for query in ["What technology does Alice use?", "Does Alice have any team preferences?"]:
    recalled = ltm.recall(query, k=2)
    print(f"\nQuery: {query}")
    for m in recalled:
        print(f"  → {m}")

---
## 4. Working Memory: Context Assembly

In [ ]:
@dataclass
class TokenBudget:
    total: int = 4000
    system_prompt: int = 500
    long_term_memory: int = 800
    conversation_history: int = 1500
    user_message: int = 500
    response_reserve: int = 700

    def allocations(self) -> dict:
        return {
            "system": self.system_prompt,
            "ltm": self.long_term_memory,
            "history": self.conversation_history,
            "user": self.user_message,
            "response": self.response_reserve,
        }


class WorkingMemory:
    """Assembles the optimal context window for each request."""

    def __init__(self, ltm: LongTermMemory, budget: TokenBudget = None):
        self.ltm = ltm
        self.budget = budget or TokenBudget()
        self.history = ConversationBuffer(max_tokens=self.budget.conversation_history)

    def assemble(self, user_message: str, system_prompt: str = "You are a helpful assistant.") -> list[dict]:
        # 1. Retrieve relevant long-term memories
        recalled = self.ltm.recall(user_message, k=3)
        ltm_context = "\n".join(f"- {m}" for m in recalled)

        # 2. Build system prompt with memory
        full_system = f"""{system_prompt}

What you know about this user:
{ltm_context}"""

        # 3. Build message list
        messages = [{"role": "system", "content": full_system}]
        messages += list(self.history._turns)   # Conversation history
        messages += [{"role": "user", "content": user_message}]
        return messages

    def chat(self, user_message: str) -> str:
        messages = self.assemble(user_message)
        response = chat(messages)
        # Store conversation turn
        self.history.add("user", user_message)
        self.history.add("assistant", response)
        # Persist important turns to long-term memory
        self.ltm.store(f"User said: {user_message[:200]}", metadata={"type": "conversation"})
        return response


wm = WorkingMemory(ltm)
print("Working memory assistant (with persistent long-term memory)\n")

for q in ["What do you know about my tech stack?", "Can you suggest a vector DB for my use case?"]:
    print(f"User: {q}")
    resp = wm.chat(q)
    print(f"Assistant: {resp[:200]}\n")

---
## 5. Memory Compression

In [ ]:
def compress_conversation(history: list[dict], target_tokens: int = 300) -> str:
    """Summarise a long conversation into a compact memory snippet."""
    turns_text = "\n".join(f"{m['role'].capitalize()}: {m['content']}" for m in history)
    original_tokens = count_tokens(turns_text)

    if original_tokens <= target_tokens:
        return turns_text  # Already short enough

    summary = chat([{
        "role": "user",
        "content": f"""Compress this conversation into a concise summary (max {target_tokens} tokens).
Keep: user's name, goals, technical preferences, important facts stated.
Drop: pleasantries, repeated information.

{turns_text}"""
    }])

    compressed_tokens = count_tokens(summary)
    print(f"Compressed: {original_tokens} → {compressed_tokens} tokens ({100*(1-compressed_tokens/original_tokens):.0f}% reduction)")
    return summary


# Simulate a long conversation to compress
long_conversation = [
    {"role": "user",      "content": "Hi! I'm Bob, a data scientist at a fintech company."},
    {"role": "assistant", "content": "Nice to meet you Bob! How can I help you today?"},
    {"role": "user",      "content": "I'm building an LLM system to analyse financial documents. We use Python, pandas, and are evaluating vector databases."},
    {"role": "assistant", "content": "That sounds like a great use case for RAG. For financial documents, Chroma or Pinecone would both work well."},
    {"role": "user",      "content": "We're leaning towards Pinecone for managed infrastructure. Budget isn't a concern."},
    {"role": "assistant", "content": "Pinecone is a solid choice for managed, scalable vector search. Their serverless tier is especially convenient."},
    {"role": "user",      "content": "We also need to handle multi-language documents — mostly English and German."},
]

compressed = compress_conversation(long_conversation)
print("\nCompressed summary:")
print(compressed)

---
## 6. Hierarchical Memory Architecture

In [ ]:
print("""
Hierarchical Memory Architecture
═════════════════════════════════

L1 — Working Memory (current request)
  • Assembled fresh each call
  • System prompt + retrieved LTM + recent history + user message
  • Lives in: the messages[] list passed to the LLM
  • Limit: your context window (128k tokens for GPT-4o)

L2 — Short-Term Memory (current session)
  • Last N turns of the conversation
  • Lives in: ConversationBuffer (in-memory deque)
  • Cleared when: session ends or token budget exceeded
  • Limit: ~1500 tokens (~10-15 turns)

L3 — Long-Term Memory (across sessions)
  • Semantic memories indexed by embedding similarity
  • Lives in: ChromaDB / Pinecone (persistent)
  • Cleared when: explicit user deletion
  • Limit: practically unlimited (millions of memories)

Flow on each request:
  New message
       │
  Query L3 (recall top-k relevant memories)
       │
  Read L2 (recent conversation turns)
       │
  Assemble L1 (system + L3 snippets + L2 history + message)
       │
  LLM call
       │
  Write: response to L2, key facts to L3
""")

---
## 🧪 Exercises

1. **Memory eviction policy**: The current `ConversationBuffer` drops the oldest turn on overflow. Implement an importance-based policy instead: score each turn with an LLM (1–5 importance), drop the lowest-scored turn when budget is exceeded.
2. **Cross-session recall**: Store 5 fake conversation facts to `LongTermMemory`, close your kernel, restart Python, and verify the memories are still there (ChromaDB persists to disk).
3. **Forgetting**: Add a `forget(keyword: str)` method to `LongTermMemory` that deletes all stored memories matching a keyword. Test that subsequent recall no longer surfaces them.
4. **Memory deduplication**: If the same fact is stored 3 times (slightly worded differently), recall returns duplicates. Add a dedup step: before storing, check if a very similar memory (cosine sim > 0.97) already exists.
5. **Full conversation**: Wire `WorkingMemory` into a terminal chat loop (like `chat_client.py` in the capstone). Observe how the model's responses change after 5+ turns vs. the first turn.

---
**Congratulations!** You've completed all 11 modules.

**Next step:** Build the [Capstone Project](../capstone/README.md) — a production app combining all modules end-to-end.

---
## 🧪 Exercises

1. **Buffer Size**: Test conversation buffers of 5, 10, 20 turns. At what point does quality degrade?
2. **Memory Retrieval**: Build vector memory with 100 past messages. Query for relevant memories.
3. **Summarization**: Compress 20 turns into a summary. Does the model remember key facts?